# Generación de imágenes de flores con Stable Diffusion

Concepto elegido: **flores**.

En este notebook se cubren los siguientes puntos:

1. Preparación de un conjunto de datos propio (`custom_dataset`) con al menos 10 imágenes de flores.
2. Configuración del entorno y comprobación de GPU.
3. Carga del modelo `stabilityai/stable-diffusion-2-1-base` y configuración de un scheduler.
4. Generación de imágenes a partir de prompts detallados, ajustando `guidance_scale` y `num_inference_steps`.
5. Generación de variaciones de una imagen base mediante image-to-image, variando `strength` y el `scheduler`.

## 1. Preparación del conjunto de datos

Descargamos 10 imágenes de flores de alta calidad desde el dataset público `huggan/flowers-102-categories` (basado en el Oxford 102 Flowers Dataset) y las organizamos en la carpeta `custom_dataset`.

In [ ]:
!pip install --upgrade diffusers transformers torch accelerate pillow datasets

In [ ]:
import os
from datasets import load_dataset

DATASET_DIR = "custom_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

flowers_dataset = load_dataset("huggan/flowers-102-categories", split="train")

NUM_IMAGES = 10

for i in range(NUM_IMAGES):
    img = flowers_dataset[i]["image"]
    img_path = os.path.join(DATASET_DIR, f"flower_{i+1:02d}.png")
    img.save(img_path)
    print(f"Guardada: {img_path}")

In [ ]:
from PIL import Image

sample_paths = [os.path.join(DATASET_DIR, f) for f in sorted(os.listdir(DATASET_DIR))[:5]]
sample_images = [Image.open(p).convert("RGB").resize((200, 200)) for p in sample_paths]

from diffusers.utils import make_image_grid
make_image_grid(sample_images, rows=1, cols=len(sample_images))

## 2. Configuración del entorno

Verificamos la disponibilidad de GPU y creamos el directorio de salida para las imágenes generadas.

In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore")

print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo utilizado: {DEVICE}")

In [ ]:
OUTPUT_DIR = "generated_outputs/"
VARIATIONS_DIR = os.path.join(OUTPUT_DIR, "variations")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VARIATIONS_DIR, exist_ok=True)

## 3. Carga del modelo y scheduler

Usamos `stabilityai/stable-diffusion-2-1-base` con `EulerAncestralDiscreteScheduler` como scheduler por defecto.

In [ ]:
from diffusers import StableDiffusionPipeline, EulerAncestralDiscreteScheduler

MODEL_ID = "stabilityai/stable-diffusion-2-1-base"

def create_pipeline(model_id=MODEL_ID, scheduler_cls=EulerAncestralDiscreteScheduler):
    scheduler = scheduler_cls.from_pretrained(model_id, subfolder="scheduler")

    torch_dtype = torch.float16 if DEVICE == "cuda" else torch.float32

    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch_dtype,
    ).to(DEVICE)

    return pipe

In [ ]:
pipe = create_pipeline()

## 4. Generación de imágenes a partir de prompts

Escribimos prompts detallados relacionados con flores y ajustamos `guidance_scale` y `num_inference_steps` para observar su efecto en la calidad.

In [ ]:
flower_prompt = "A macro photo of a vibrant red rose in full bloom, dew drops on petals, soft natural light, high detail, professional photography"
flower_negative_prompt = "blurry, low quality, deformed, watermark, text, cropped"

generator = torch.Generator(device=DEVICE).manual_seed(42)

flower_image = pipe(
    flower_prompt,
    negative_prompt=flower_negative_prompt,
    guidance_scale=7.5,
    num_inference_steps=50,
    height=512,
    width=512,
    generator=generator,
).images[0]

flower_image_path = os.path.join(OUTPUT_DIR, "rose_base.png")
flower_image.save(flower_image_path)

flower_image

Probamos otro prompt (tulipanes) variando `guidance_scale` y `num_inference_steps` para comparar el efecto en el resultado.

In [ ]:
tulip_prompt = "A field of colorful tulips at sunrise, wide angle, vibrant colors, cinematic lighting, highly detailed"
tulip_negative_prompt = "blurry, low quality, deformed, watermark, text, cropped"

generator = torch.Generator(device=DEVICE).manual_seed(84)

tulip_image_low = pipe(
    tulip_prompt,
    negative_prompt=tulip_negative_prompt,
    guidance_scale=4,
    num_inference_steps=25,
    height=512,
    width=512,
    generator=generator,
).images[0]

generator = torch.Generator(device=DEVICE).manual_seed(84)

tulip_image_high = pipe(
    tulip_prompt,
    negative_prompt=tulip_negative_prompt,
    guidance_scale=14,
    num_inference_steps=80,
    height=512,
    width=512,
    generator=generator,
).images[0]

tulip_image_low.save(os.path.join(OUTPUT_DIR, "tulips_low_params.png"))
tulip_image_high.save(os.path.join(OUTPUT_DIR, "tulips_high_params.png"))

make_image_grid([tulip_image_low, tulip_image_high], rows=1, cols=2)

## 5. Variaciones de imágenes existentes (Image-to-Image)

Usamos una de las imágenes del `custom_dataset` como imagen base y generamos variaciones guiadas por texto, cambiando `strength` y el `scheduler`.

In [ ]:
from diffusers import AutoPipelineForImage2Image, DDIMScheduler

img2img_pipe = AutoPipelineForImage2Image.from_pipe(pipe)

base_image_path = os.path.join(DATASET_DIR, "flower_01.png")
base_image = Image.open(base_image_path).convert("RGB").resize((512, 512))

base_image

In [ ]:
variation_prompt = "A futuristic version of the flower with neon accents and a dark background, glowing petals, sci-fi style"

generator = torch.Generator(device=DEVICE).manual_seed(7)

variation_strength_low = img2img_pipe(
    prompt=variation_prompt,
    image=base_image,
    strength=0.4,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=generator,
).images[0]

variation_strength_low_path = os.path.join(VARIATIONS_DIR, "flower_neon_strength_04.png")
variation_strength_low.save(variation_strength_low_path)

variation_strength_low

In [ ]:
generator = torch.Generator(device=DEVICE).manual_seed(7)

variation_strength_high = img2img_pipe(
    prompt=variation_prompt,
    image=base_image,
    strength=0.8,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=generator,
).images[0]

variation_strength_high_path = os.path.join(VARIATIONS_DIR, "flower_neon_strength_08.png")
variation_strength_high.save(variation_strength_high_path)

variation_strength_high

Cambiamos ahora el scheduler del pipeline de image-to-image (`DDIMScheduler`) manteniendo el mismo prompt y `strength`, para comparar cómo afecta el scheduler al resultado.

In [ ]:
img2img_pipe.scheduler = DDIMScheduler.from_config(img2img_pipe.scheduler.config)

generator = torch.Generator(device=DEVICE).manual_seed(7)

variation_ddim = img2img_pipe(
    prompt=variation_prompt,
    image=base_image,
    strength=0.6,
    guidance_scale=7.5,
    num_inference_steps=50,
    generator=generator,
).images[0]

variation_ddim_path = os.path.join(VARIATIONS_DIR, "flower_neon_ddim_scheduler.png")
variation_ddim.save(variation_ddim_path)

variation_ddim

Comparamos la imagen base junto a las tres variaciones (strength bajo, strength alto y scheduler distinto) en una sola cuadrícula.

In [ ]:
comparison_grid = make_image_grid(
    [base_image, variation_strength_low, variation_strength_high, variation_ddim],
    rows=1,
    cols=4,
)

comparison_grid_path = os.path.join(VARIATIONS_DIR, "flower_variations_comparison_grid.png")
comparison_grid.save(comparison_grid_path)

comparison_grid

## Conclusiones

- Se construyó un `custom_dataset` con 10 imágenes de flores obtenidas del dataset público `huggan/flowers-102-categories`.
- Se configuró un pipeline de Stable Diffusion (`stable-diffusion-2-1-base`) con `EulerAncestralDiscreteScheduler`.
- Se generaron imágenes de flores a partir de prompts detallados, observando el efecto de `guidance_scale` y `num_inference_steps` en la calidad y fidelidad al prompt.
- Se generaron variaciones de una imagen base del dataset mediante image-to-image, comprobando cómo `strength` controla cuánto se aleja el resultado de la imagen original, y cómo el `scheduler` elegido afecta también al resultado final.
- Todas las imágenes generadas se guardaron en `generated_outputs/` (imágenes base) y `generated_outputs/variations/` (variaciones image-to-image).